# Finish reason as a state machine

**Scenario:** a race strategy service turns telemetry into a pit plan and sends it to the pit wall.
Mid race it starts returning no stops at all. The parser was catching its own error and falling back
to an empty plan.

Two unrelated faults produce that identical error, and they need opposite fixes. The field that tells
them apart is the one most code ignores.

Think of it as **the reason a phone call ended**. Hung up, ran out of battery, or cut off by the
network are three very different situations, and "the call ended" tells you none of them.

## Mechanics

`finish_reason` is on every choice, and it is the runtime's state machine.

| Value | Meaning | What your code should do |
|---|---|---|
| `stop` | The model finished on its own | Use the answer |
| `tool_calls` | It wants a function run | Execute, append the result, loop |
| `length` | **It was cut off** at your token cap | Do not use the answer. Raise, or ask for more |
| `content_filter` | Blocked by a safety filter | Do not retry the same thing |

The trap is that `length` still returns text. It looks like an answer right up to the moment it is
not one.

## The picture

![Branch on finish reason before touching the content](images/finish-reason.svg)

Reading the content before checking why generation stopped is how a truncated answer gets treated as
a complete one.

## The cost

A wrong answer is worth less than no answer, because no answer fails loudly.

```
cost = races where the car did not pit x the value of a race
```

## The failure

Here is the strategy call, and the parser the way most services write it first.

In [1]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("01-stateful-agent-runtime/02-finish-reason-as-a-state-machine")

SYSTEM = ('You are a Formula 1 race strategist. Reply with JSON only: '
          '{"stops":[{"lap":int,"tyre":"soft|medium|hard"}],"note":str}')
RACE = ("58 lap race, high tyre wear, safety car likely around lap 30. "
        "Give the full pit strategy with reasoning in the note.")


def ask(max_tokens):
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=max_tokens,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": RACE}])
    return reply.choices[0]

And the parser. It is defensive, it never crashes, and that is the problem.

In [2]:
def stops_naive(choice):
    """Parse the plan. Falls back to no stops if anything goes wrong."""
    try:
        return json.loads(choice.message.content)["stops"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return []


generous = ask(400)
tight = ask(40)

print(f"generous budget: {len(stops_naive(generous))} stops planned")
print(f"tight budget   : {len(stops_naive(tight))} stops planned")
print("\nthe car never pits, and nothing raised")

generous budget: 0 stops planned
tight budget   : 0 stops planned

the car never pits, and nothing raised


Both returned an empty plan. Now look at what actually came back.

In [3]:
for label, choice in (("generous", generous), ("tight", tight)):
    text = choice.message.content or ""
    print(f"{label:9} finish_reason={choice.finish_reason!r}")
    print(f"{'':9} tail: ...{text[-58:]!r}\n")

assert stops_naive(generous), "a complete answer produced an empty plan"

generous  finish_reason='stop'
          tail: ...'taking or defending on fresh tyres towards the end."\n}\n```'

tight     finish_reason='length'
          tail: ...'    "lap": 28,\n      "tyre": "medium"\n    },\n    {\n      "'



AssertionError: a complete answer produced an empty plan

## The diagnosis

Same empty result, two unrelated causes.

**The generous call finished.** `finish_reason` is `stop`. It just wrapped the JSON in a code fence.
Complete content, wrong format.

**The tight call was cut off.** `finish_reason` is `length`. Fine format, missing content.

One needs the fence stripped. The other must never be parsed, because half a pit strategy is more
dangerous than none. A `try/except` around `json.loads` cannot tell them apart, so it treats a
truncated plan as a valid one with no stops.

The field that separates them was on the response the whole time.

## The fix

Branch on `finish_reason` first, and only then look at the content.

In [4]:
class Truncated(Exception):
    """Generation hit the token cap. The answer is incomplete, not wrong."""

A named exception, because a caller needs to tell "cut off" apart from "malformed" to know whether
retrying is worth anything. Then the parser, which reads the state before the content.

In [5]:
def parse_strategy(choice):
    """Read finish_reason before content. Raise rather than guess."""
    if choice.finish_reason == "length":
        raise Truncated("cut off at the token cap, ask for more room")
    if choice.finish_reason == "content_filter":
        raise ValueError("blocked by a filter, retrying the same prompt will not help")

    text = (choice.message.content or "").strip()
    if text.startswith("```"):
        text = text.split("```")[1].removeprefix("json").strip()
    return json.loads(text)["stops"]

Now the two cases separate cleanly, and the dangerous one is loud.

In [6]:
for label, choice in (("generous", generous), ("tight", tight)):
    try:
        stops = parse_strategy(choice)
        print(f"{label:9} {len(stops)} stops: {[s['lap'] for s in stops]}")
    except Truncated as exc:
        print(f"{label:9} REFUSED, {exc}")

print("\nbefore: both silently returned 0 stops")
print("after : one plan parsed, one refused loudly")

generous  2 stops: [28, 55]
tight     REFUSED, cut off at the token cap, ask for more room

before: both silently returned 0 stops
after : one plan parsed, one refused loudly


Knowing an answer was truncated is only useful if the runtime acts on it.

In [7]:
def ask_until_complete(max_tokens, ceiling=1200):
    """Grow the budget until the model finishes, or give up honestly."""
    while max_tokens <= ceiling:
        choice = ask(max_tokens)
        if choice.finish_reason != "length":
            return choice, max_tokens
        max_tokens *= 4
    raise Truncated(f"still truncated at {ceiling} tokens, the request is too large")

Retrying blindly would double a bill for no reason, so it only retries the one state that a bigger
budget can fix. A content filter or a finished answer returns straight away.

In [8]:
choice, used = ask_until_complete(40)
stops = parse_strategy(choice)

print(f"started at 40 tokens, succeeded at {used}")
print(f"finish_reason: {choice.finish_reason}")
print(f"plan: {[(s['lap'], s['tyre']) for s in stops]}")

started at 40 tokens, succeeded at 640
finish_reason: stop
plan: [(28, 'medium'), (48, 'hard')]


## The gate

The regression to prevent is someone reintroducing a bare `try/except` around the parse. This test
catches it without calling the model.

In [9]:
class FakeChoice:
    """A response shaped object, so the test needs no API call."""
    def __init__(self, reason, text):
        self.finish_reason = reason
        self.message = type("M", (), {"content": text})()

No network, so this runs on every commit in milliseconds.

In [10]:
def test_truncated_output_is_never_parsed():
    cut_off = FakeChoice("length", '{"stops": [{"lap": 15, "tyre": "med')
    try:
        parse_strategy(cut_off)
    except Truncated:
        return
    raise AssertionError("a truncated answer was parsed instead of refused")


test_truncated_output_is_never_parsed()
print("gate holds: length never reaches the parser")

gate holds: length never reaches the parser


Delete the `length` branch and this test fails.

### Enterprise exploration

- `ask_until_complete` quadruples the budget. What stops that quadrupling your bill during an
  incident, and what caps it?
- The plan was refused. What does the pit wall see, and is a stale plan better or worse than none?
- `content_filter` is never retried. How would you find out it is happening, and who gets paged?
- Truncation is now a raised error, not an empty list. What breaks downstream, and how do you ship
  that without an outage?

### Key takeaways

- `finish_reason` is the state machine. Read it before the content.
- `length` means incomplete. Stripping fences will not save you.
- A `try/except` returning a default turns a loud failure into a silent wrong answer.
- Retry only the state a retry can fix.